# BD3 Building Defect Detection — Colab Training

**Before running:**
1. Pack the project locally:  `./pack_dataset.sh` → produces `bd3_cloud_pkg.tar`
2. Upload `bd3_cloud_pkg.tar` to your Google Drive (root or `MyDrive/bd3/`)
3. **Runtime → Change runtime type → GPU** (T4 free; A100/L4 on Colab Pro)
4. Run cells top to bottom.

Best.pt and the full `runs/` directory are copied back to Drive at the end.

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. GPU sanity check

In [ ]:
!nvidia-smi

## 3. Locate the tarball on Drive
Edit `TAR_ON_DRIVE` if you put it elsewhere.

In [ ]:
import os
TAR_ON_DRIVE = '/content/drive/MyDrive/bd3_cloud_pkg.tar'  # change if needed
RUN_NAME     = 'bd3_yolo11s_v1'
DRIVE_OUT    = '/content/drive/MyDrive/bd3_runs'

assert os.path.exists(TAR_ON_DRIVE), f'tarball not found: {TAR_ON_DRIVE}'
os.makedirs(DRIVE_OUT, exist_ok=True)
print('tar size:', os.path.getsize(TAR_ON_DRIVE) / 1e9, 'GB')

## 4. Extract to `/content/bd3` (local SSD — much faster than Drive for training I/O)

In [ ]:
!mkdir -p /content/bd3 && tar -xf "$TAR_ON_DRIVE" -C /content/bd3 && ls /content/bd3

## 5. Patch `data.yaml` to point at the Colab path
Your local `data.yaml` has a hardcoded `/home/shalini/...` path.

In [ ]:
import yaml
DATA_YAML = '/content/bd3/data.yaml'
with open(DATA_YAML) as f:
    cfg = yaml.safe_load(f)
cfg['path'] = '/content/bd3/dataset'
with open(DATA_YAML, 'w') as f:
    yaml.safe_dump(cfg, f, sort_keys=False)
print(cfg)

## 6. Install dependencies

In [ ]:
%pip install -q --upgrade ultralytics albumentations

## 7. Quick dataset re-validation (catches upload corruption)

In [ ]:
%cd /content/bd3
!python validate_dataset.py --root dataset --num-classes 6 --workers 4 \
    --report /content/bd3/cloud_validate_report.json | tail -25

## 8. Train
**Default:** yolo11s @ 640px, 200 epochs, batch 32, AMP on. Total time on a T4: ~6–10h. On L4/A100: ~2–4h.

Uncomment the smaller config first if you want a 1–2h smoke run on a free T4 to verify.

In [ ]:
%cd /content/bd3

# --- Smoke run (uncomment for a 30-60min sanity check on free T4) ---
# !python train_yolo.py --model yolo11n.pt --epochs 5 --imgsz 480 \
#     --batch 32 --device 0 --workers 4 --name smoketest --patience 999

# --- Real training run ---
!python train_yolo.py \
    --model yolo11s.pt \
    --epochs 200 \
    --imgsz 640 \
    --batch 32 \
    --device 0 \
    --workers 4 \
    --patience 30 \
    --save-period 25 \
    --name $RUN_NAME

## 9. Evaluate

In [ ]:
BEST = f'/content/bd3/runs/detect/{RUN_NAME}/weights/best.pt'
!python validate_and_compare.py --weights $BEST --data /content/bd3/data.yaml --imgsz 640 --device 0

## 10. Copy results back to Drive
Saves: `best.pt`, `last.pt`, plots, results.csv, confusion_matrix.png — everything you'll want to inspect locally.

In [ ]:
import shutil
src = f'/content/bd3/runs/detect/{RUN_NAME}'
dst = f'{DRIVE_OUT}/{RUN_NAME}'
if os.path.isdir(dst):
    shutil.rmtree(dst)
shutil.copytree(src, dst)
print('Copied results to:', dst)
print('Files:')
for f in sorted(os.listdir(dst)):
    print('  ', f)

## 11. (Optional) Export to ONNX / OpenVINO for fast CPU inference

In [ ]:
from ultralytics import YOLO
m = YOLO(BEST)
# ONNX is the most portable; OpenVINO is fastest on Intel CPU.
m.export(format='onnx', imgsz=640, half=False, simplify=True)
# m.export(format='openvino', imgsz=640, half=False)
import shutil
for ext in ('.onnx', '_openvino_model'):
    src = BEST.replace('.pt', ext)
    if os.path.exists(src):
        dst = f'{DRIVE_OUT}/{RUN_NAME}/{os.path.basename(src)}'
        if os.path.isdir(src):
            shutil.copytree(src, dst, dirs_exist_ok=True)
        else:
            shutil.copy2(src, dst)
        print('exported ->', dst)